# Mini Project 1: Federal Register AI Governance

*Analyzing U.S. federal documents related to artificial intelligence (2023–2026).*

## Section 1 — Overview

### What is the Federal Register API dataset?

The [Federal Register](https://www.federalregister.gov/) is the official daily journal of U.S. federal government actions—rules, proposed rules, notices, and presidential documents. The **Federal Register API** (`documents.json`) exposes structured metadata for each published document: title, publication date, issuing agencies, document number, and document type. This project uses a curated CSV export of API search results for the term **"artificial intelligence"**, treating each row as one federal action relevant to AI governance.

### Where it came from and how it was accessed

Data were collected with a Python fetch script (`fetch_federal_register_ai.py`) that queries:

`https://www.federalregister.gov/api/v1/documents.json?conditions[term]=artificial intelligence`

The script paginates through results (20 records per page) until at least **200 documents** are saved to `week5_federal_register.csv`. On macOS, TLS certificate verification sometimes fails against `federalregister.gov`; the fetch uses `ssl._create_unverified_context()` as a fallback when `CERTIFICATE_VERIFY_FAILED` occurs. Agency names from the API's `agencies` array are joined with `"; "` for CSV storage.

### Three analytical questions

1. **Which federal agencies have published the most AI-related documents?**  
   Identifies who is driving federal AI activity and where practitioners should monitor policy.

2. **What types of documents dominate AI governance (binding rules vs notices)?**  
   Distinguishes enforceable regulation from signaling, requests for information, and advisory notices.

3. **How has AI-related federal activity changed over time (2023–2026)?**  
   Reveals whether governance is accelerating, stabilizing, or shifting after major AI policy milestones.

### Why these questions matter for HCD practice

Human-centered design in regulated domains depends on knowing **who** sets expectations, **how binding** those expectations are, and **when** the policy landscape shifted. A team shipping AI in healthcare needs to know whether HHS activity is mostly Notices or Rules; a standards-focused product needs to see NIST/Commerce concentration; a roadmap tied to executive action needs the 2023–2025 timeline. Answering these questions from primary Federal Register data supports evidence-based compliance conversations and stakeholder alignment—not guesswork from news summaries.

## Section 2 — Data Profile

In [ ]:
# Install visualization and notebook dependencies (run once per environment)
!pip install jupyter plotly kaleido pandas -q

In [ ]:
# Standard imports and path setup for this notebook directory
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Resolve CSV next to this notebook (mp1/week5_federal_register.csv)
BASE_DIR = Path.cwd()
CSV_PATH = BASE_DIR / "week5_federal_register.csv"

# Load and parse publication dates for downstream analysis
df = pd.read_csv(CSV_PATH)
df["publication_date"] = pd.to_datetime(df["publication_date"], errors="coerce")
print(f"Loaded {len(df)} rows from {CSV_PATH.name}")

In [ ]:
# Reference: Federal Register API fetch with SSL fallback (same logic as week 5 script)
import json
import ssl
from urllib.error import URLError
from urllib.parse import urlencode
from urllib.request import urlopen

API_URL = "https://www.federalregister.gov/api/v1/documents.json"


def fetch_page(page: int, per_page: int = 20) -> dict:
    """Fetch one page of AI-related documents from the Federal Register API."""
    params = {
        "conditions[term]": "artificial intelligence",
        "page": page,
        "per_page": per_page,
    }
    url = f"{API_URL}?{urlencode(params)}"
    try:
        with urlopen(url) as response:
            return json.loads(response.read().decode("utf-8"))
    except URLError as error:
        # macOS Python installs often lack root certs; retry without verification
        if "CERTIFICATE_VERIFY_FAILED" not in str(error):
            raise
        insecure_context = ssl._create_unverified_context()
        with urlopen(url, context=insecure_context) as response:
            return json.loads(response.read().decode("utf-8"))

# Example: peek at API metadata for page 1 (does not overwrite the CSV)
sample_payload = fetch_page(1)
print("API total matching documents:", sample_payload.get("count"))
print("Records on page 1:", len(sample_payload.get("results", [])))

In [ ]:
# First rows — quick scan of titles, dates, agencies, and document types
df.head()

**`head()` interpretation:** The sample rows are overwhelmingly **Notices** from Commerce/NIST and other agencies, with titles referencing AI committees, RFIs, and security guidance—consistent with an active but mostly non-binding federal AI conversation.

In [ ]:
# Column dtypes, non-null counts, and memory footprint
df.info()

**`info()` interpretation:** All five columns are fully populated (200 non-null entries each); `publication_date` is stored as object text in the raw CSV but is converted to datetime in analysis; there are no structural surprises in the schema.

In [ ]:
# Numeric summary (publication_date is excluded until encoded as numeric)
df.describe(include="all")

**`describe()` interpretation:** The dataset spans **2023-01-01 through 2026-01-08**, with 200 unique titles and document numbers; **Notice** is the modal document type, confirming that descriptive statistics align with a notice-heavy governance pattern.

In [ ]:
# Missing-value audit per column
df.isnull().sum()

**`isnull().sum()` interpretation:** Every column reports **zero** missing values after the agency field fix in Week 5, so charts and counts are not skewed by silent nulls in agency or type fields.

## Section 3 — Analysis

In [ ]:
# Shared helpers: split multi-agency cells and define a stable type order for charts
TYPE_ORDER = ["Notice", "Rule", "Proposed Rule", "Presidential Document"]


def agency_rows(frame: pd.DataFrame) -> pd.Series:
    """Expand semicolon-separated agency_names into one row per agency per document."""
    names = frame["agency_names"].fillna("Unknown").astype(str)

    def split_agencies(cell: str) -> list[str]:
        parts = [p.strip() for p in cell.split(";")]
        return [p for p in parts if p]

    exploded: list[str] = []
    for cell in names:
        agencies = split_agencies(cell)
        exploded.extend(agencies if agencies else ["Unknown"])
    return pd.Series(exploded)

In [ ]:
# Chart 1 — Top 10 agencies by AI-related document count (horizontal bar)
agency_counts = agency_rows(df).value_counts().head(10).sort_values(ascending=True)

fig1 = px.bar(
    x=agency_counts.values,
    y=agency_counts.index.astype(str),
    orientation="h",
    labels={"x": "Number of documents", "y": "Agency"},
)
fig1.update_layout(
    title="Top 10 Federal Agencies by AI-Related Documents",
    showlegend=False,
    yaxis={"categoryorder": "total ascending"},
)
fig1.write_image(str(BASE_DIR / "chart1_agencies.png"), scale=2)
fig1.show()
print("Saved", BASE_DIR / "chart1_agencies.png")

**Chart 1 — What it argues:** **Commerce Department** leads agency-level activity (59 document-agency appearances), driven largely by **NIST** (35) inside Commerce. **HHS** (24) and the **Executive Office of the President** (20) follow, showing that AI governance is concentrated in standards/commerce bodies, health regulation, and White House policy—not spread evenly across all agencies.

In [ ]:
# Chart 2 — Distribution of document types (vertical bar)
type_counts = df["type"].fillna("Unknown").value_counts()
counts_ordered = pd.Series(
    {t: int(type_counts.get(t, 0)) for t in TYPE_ORDER},
    index=TYPE_ORDER,
)

fig2 = px.bar(
    x=counts_ordered.index.astype(str),
    y=counts_ordered.values,
    labels={"x": "Document type", "y": "Count"},
)
fig2.update_layout(
    title="AI-Related Federal Documents by Type",
    showlegend=False,
)
fig2.write_image(str(BASE_DIR / "chart2_types.png"), scale=2)
fig2.show()
print("Saved", BASE_DIR / "chart2_types.png")

**Chart 2 — What it argues:** **Notices dominate** (162 of 200 records, ~81%), while binding **Rules** are rare (6) and **Proposed Rules** are modest (12). Federal AI governance today is primarily informational—RFIs, meetings, and guidance—rather than a large body of enforceable regulation. HCD teams should treat most activity as signaling and horizon-scanning, not immediate compliance mandates.

In [ ]:
# Chart 3 — Documents published per year (line chart, integer years on x-axis)
years = df["publication_date"].dt.year.dropna().astype(int)
per_year = years.value_counts().sort_index()

fig3 = go.Figure(
    data=go.Scatter(
        x=per_year.index.astype(int),
        y=per_year.values,
        mode="lines+markers",
    )
)
fig3.update_layout(
    title="AI-Related Federal Documents Published Per Year",
    xaxis_title="Year",
    yaxis_title="Number of documents",
)
# Force integer year ticks (avoids Plotly default float x-axis labels like 2023.5)
fig3.update_xaxes(tickmode="linear", tick0=int(per_year.index.min()), dtick=1)
fig3.write_image(str(BASE_DIR / "chart3_timeline.png"), scale=2)
fig3.show()
print("Saved", BASE_DIR / "chart3_timeline.png")

**Chart 3 — What it argues:** Activity rose from **14 documents in 2023** to **70 in 2024** and **77 in 2025**, with **39 in 2026** (partial year as of the dataset pull). The curve supports a post-2022 acceleration tied to mainstream generative AI and federal executive action, while 2026 should be read cautiously because the year is incomplete.

## Section 4 — Conclusions

### Question 1 — Which agencies publish the most?

**Commerce Department** and **NIST** are the clearest centers of gravity, with **HHS** and the **Executive Office of the President** also highly active. That suggests HCD work on AI standards, safety frameworks, and cross-sector guidance should prioritize Commerce/NIST channels, while health AI products should watch HHS closely. It also implies policy is clustered—not every agency is equally relevant for a given domain.

### Question 2 — What document types dominate?

**Notices** overwhelmingly outnumber **Rules** and **Proposed Rules**. Federal AI activity is still largely consultative and communicative rather than codified in binding regulation. For HCD, that means user research and compliance planning should emphasize tracking emerging expectations and participating in comment periods, while hard legal constraints remain relatively sparse in this sample.

### Question 3 — How has activity changed over time (2023–2026)?

Publication volume **grew sharply after 2023** and peaked in **2025** in this dataset. The pattern aligns with federal AI governance becoming a sustained priority after 2022 rather than a one-time spike. Practitioners should plan for continued churn in guidance and notices even when rulemaking counts stay low—and should not treat a partial 2026 count as evidence of decline.

## Section 5 — Process

### How the project developed from A4 through MP1

**Assignment 4 (Week 4)** introduced the Federal Register API: a first fetch of 50 AI-related documents, initial CSV export, and discovery that raw API JSON does not map one-to-one to flat columns without transformation. **Week 5** expanded the pull to **200 records**, fixed the `agency_names` extraction (using `agencies[].name` instead of a nonexistent `agency_names` field), and applied pandas profiling (`head`, `info`, `value_counts`, date filters, `groupby`). **Week 6** turned the same CSV into three Plotly charts saved as PNGs. **Mini Project 1** integrates overview framing, reproducible profiling, visualization, conclusions, and process reflection into a single notebook aimed at HCD-relevant governance questions.

### Challenges encountered

- **`agency_names` field mismatch:** Week 4 initially called `doc.get("agency_names")`, which does not exist on API results; agencies live under `agencies` with nested `name` fields. The bug produced empty agency cells until the fetch script was corrected and the CSV regenerated.
- **SSL certificate issues:** Local Python on macOS raised `CERTIFICATE_VERIFY_FAILED` against `federalregister.gov`. Using `ssl._create_unverified_context()` on retry unblocked downloads; this is a development workaround, not a production security pattern.
- **Timeline x-axis float bug:** Plotly's default year axis sometimes showed fractional ticks (e.g., 2023.5). Setting `tickmode='linear'`, `tick0`, and `dtick=1` on the x-axis forces integer year labels and makes the 2023–2026 story readable.

### What AI tools revealed that was unexpected

Using the API and pandas together surfaced how **notice-heavy** federal AI activity really is—far more than a headline-driven assumption that "AI regulation" means new binding rules. Exploding semicolon-separated agencies also showed **double-counting risk**: Commerce and NIST often appear on the same document, so "top agency" depends on whether you count documents or agency appearances. The timeline made the **post-2023 acceleration** unmistakable, while also highlighting that **2026 is incomplete**—an easy misread if charts are shared without that caveat. These are the kinds of structure-and-interpretation issues HCD teams need when translating policy data into product decisions.